## Objective

Clean and preprocess each Olist table independently (no merges), then export the cleaned datasets to ```/data/processed/``` for reuse in later analysis.

In [135]:
import numpy as np
import pandas as pd

In [136]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [137]:
customers = pd.read_csv("/content/drive/MyDrive/original/olist_customers_dataset.csv")
orders = pd.read_csv("/content/drive/MyDrive/original/olist_orders_dataset.csv")
items = pd.read_csv("/content/drive/MyDrive/original/olist_order_items_dataset.csv")
payments = pd.read_csv("/content/drive/MyDrive/original/olist_order_payments_dataset.csv")
reviews = pd.read_csv("/content/drive/MyDrive/original/olist_order_reviews_dataset.csv")
geolocations = pd.read_csv("/content/drive/MyDrive/original/olist_geolocation_dataset.csv")
sellers = pd.read_csv("/content/drive/MyDrive/original/olist_sellers_dataset.csv")
products = pd.read_csv("/content/drive/MyDrive/original/olist_products_dataset.csv")
name_translations = pd.read_csv("/content/drive/MyDrive/original/product_category_name_translation.csv")

In [138]:
data = [customers , orders , items , payments , reviews , geolocations , sellers , products , name_translations]
table_names = ["Customers", "Orders", "Items", "Payments", "Reviews", "Geolocations", "Sellers", "Products", "Name Translations"]

In [139]:
for table in data:
  print(table.columns)
  print()

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')

Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')

Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='object')

Index(['seller_id', 

In [140]:
for name, df in zip(table_names, data):
    print(f"\n{'='*40}")
    print(f"Table: {name}")
    print(f"{'='*40}")
    df.info()
    print("-" * 40)


Table: Customers
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
----------------------------------------

Table: Orders
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-

In [141]:
# unique status values
status_col = [customers["customer_state"],orders["order_status"],payments["payment_type"],reviews["review_score"],geolocations["geolocation_state"],sellers["seller_state"]]

In [142]:
for col in status_col:
    print(f"Column: {col.name}")
    print(f"Value Counts:\n{col.value_counts()}")
    print("-" * 30)

Column: customer_state
Value Counts:
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64
------------------------------
Column: order_status
Value Counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
------------------------------
Column: payment_type
Value Counts:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64
------------------------------
Column: review_score
Value Counts:
review_score
5    57328
4    19142
1    11424
3  

In [143]:
date_cols = ["order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date"]

In [144]:
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')
orders[date_cols].dtypes

,0
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


In [145]:
for name, df in zip(table_names, data):
  print(f"\n{'='*40}")
  print(f"Table: {name}")
  print(f"{'='*40}")
  print("Null Counts and Percentages:")
  for column in df.columns:
    null_count = df[column].isnull().sum()
    if null_count > 0:
      null_percentage = (df[column].isnull().mean() * 100).round(2)
      print(f"  {column}: {null_count} ({null_percentage}%)")
    else:
      print(f"  {column}: {null_count} (0.0%)")
  print("-" * 40)


Table: Customers
Null Counts and Percentages:
  customer_id: 0 (0.0%)
  customer_unique_id: 0 (0.0%)
  customer_zip_code_prefix: 0 (0.0%)
  customer_city: 0 (0.0%)
  customer_state: 0 (0.0%)
----------------------------------------

Table: Orders
Null Counts and Percentages:
  order_id: 0 (0.0%)
  customer_id: 0 (0.0%)
  order_status: 0 (0.0%)
  order_purchase_timestamp: 0 (0.0%)
  order_approved_at: 160 (0.16%)
  order_delivered_carrier_date: 1783 (1.79%)
  order_delivered_customer_date: 2965 (2.98%)
  order_estimated_delivery_date: 0 (0.0%)
----------------------------------------

Table: Items
Null Counts and Percentages:
  order_id: 0 (0.0%)
  order_item_id: 0 (0.0%)
  product_id: 0 (0.0%)
  seller_id: 0 (0.0%)
  shipping_limit_date: 0 (0.0%)
  price: 0 (0.0%)
  freight_value: 0 (0.0%)
----------------------------------------

Table: Payments
Null Counts and Percentages:
  order_id: 0 (0.0%)
  payment_sequential: 0 (0.0%)
  payment_type: 0 (0.0%)
  payment_installments: 0 (0.0%)
 

In [146]:
orders.dropna(axis = 0, inplace = True)
products.dropna(axis = 0, inplace = True)
reviews.drop(['review_comment_title', 'review_comment_message'], axis = 1, inplace = True)

In [147]:
for name, df in zip(table_names, data):
  print(f"\n{'='*40}")
  print(f"Table: {name}")
  print(f"{'='*40}")
  print("Null Counts and Percentages:")
  for column in df.columns:
    null_count = df[column].isnull().sum()
    if null_count > 0:
      null_percentage = (df[column].isnull().mean() * 100).round(2)
      print(f"  {column}: {null_count} ({null_percentage}%)")
    else:
      print(f"  {column}: {null_count} (0.0%)")
  print("-" * 40)


Table: Customers
Null Counts and Percentages:
  customer_id: 0 (0.0%)
  customer_unique_id: 0 (0.0%)
  customer_zip_code_prefix: 0 (0.0%)
  customer_city: 0 (0.0%)
  customer_state: 0 (0.0%)
----------------------------------------

Table: Orders
Null Counts and Percentages:
  order_id: 0 (0.0%)
  customer_id: 0 (0.0%)
  order_status: 0 (0.0%)
  order_purchase_timestamp: 0 (0.0%)
  order_approved_at: 0 (0.0%)
  order_delivered_carrier_date: 0 (0.0%)
  order_delivered_customer_date: 0 (0.0%)
  order_estimated_delivery_date: 0 (0.0%)
----------------------------------------

Table: Items
Null Counts and Percentages:
  order_id: 0 (0.0%)
  order_item_id: 0 (0.0%)
  product_id: 0 (0.0%)
  seller_id: 0 (0.0%)
  shipping_limit_date: 0 (0.0%)
  price: 0 (0.0%)
  freight_value: 0 (0.0%)
----------------------------------------

Table: Payments
Null Counts and Percentages:
  order_id: 0 (0.0%)
  payment_sequential: 0 (0.0%)
  payment_type: 0 (0.0%)
  payment_installments: 0 (0.0%)
  payment_va

In [148]:
orders = orders[~orders['order_status'].isin(['canceled', 'unavailable'])]

In [149]:
for name, df in zip(table_names, data):
  print(f"\n{'='*40}")
  print(f"Table: {name}")
  print(f"{'='*40}")
  print(df.dtypes)
  print("-" * 40)


Table: Customers
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object
----------------------------------------

Table: Orders
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object
----------------------------------------

Table: Items
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object
----------------------------------------

Table: Paym

In [150]:
print('Checking for duplicates in ID columns...')
for name, df in zip(table_names, data):
    print(f"\n{'='*40}")
    print(f"Table: {name}")
    print(f"{'='*40}")
    id_columns = [col for col in df.columns if 'id' in col.lower()]
    if not id_columns:
        print("No ID columns found in this table.")
        print("-" * 40)
        continue

    found_duplicates = False
    for col in id_columns:
        duplicates_count = df[col].duplicated().sum()
        if duplicates_count > 0:
            print(f"Column '{col}': {duplicates_count} duplicate(s) found.")
            found_duplicates = True
    if not found_duplicates:
        print("No duplicates found in ID columns.")
    print("-" * 40)

Checking for duplicates in ID columns...

Table: Customers
Column 'customer_unique_id': 3345 duplicate(s) found.
----------------------------------------

Table: Orders
No duplicates found in ID columns.
----------------------------------------

Table: Items
Column 'order_id': 13984 duplicate(s) found.
Column 'order_item_id': 112629 duplicate(s) found.
Column 'product_id': 79699 duplicate(s) found.
Column 'seller_id': 109555 duplicate(s) found.
----------------------------------------

Table: Payments
Column 'order_id': 4446 duplicate(s) found.
----------------------------------------

Table: Reviews
Column 'review_id': 814 duplicate(s) found.
Column 'order_id': 551 duplicate(s) found.
----------------------------------------

Table: Geolocations
No ID columns found in this table.
----------------------------------------

Table: Sellers
No duplicates found in ID columns.
----------------------------------------

Table: Products
Column 'product_width_cm': 32245 duplicate(s) found.
-----

In [151]:
print('Checking delivery date vs. purchase date...')
discrepancies_delivery = orders[orders['order_delivered_customer_date'] < orders['order_purchase_timestamp']]
if not discrepancies_delivery.empty:
    print(f"Found {len(discrepancies_delivery)} orders where delivery date is before purchase date:")
    print(discrepancies_delivery[['order_id', 'order_purchase_timestamp', 'order_delivered_customer_date']])
else:
    print("No orders found where delivery date is before purchase date. Data is consistent.")

Checking delivery date vs. purchase date...
No orders found where delivery date is before purchase date. Data is consistent.


In [152]:
print('\nChecking approval date vs. purchase date...')
discrepancies_approved = orders[orders['order_approved_at'] < orders['order_purchase_timestamp']]
if not discrepancies_approved.empty:
    print(f"Found {len(discrepancies_approved)} orders where approval date is before purchase date:")
    print(discrepancies_approved[['order_id', 'order_purchase_timestamp', 'order_approved_at']])
else:
    print("No orders found where approval date is before purchase date. Data is consistent.")


Checking approval date vs. purchase date...
No orders found where approval date is before purchase date. Data is consistent.


In [153]:
problematic_prices = items[items['price'] <= 0]
if not problematic_prices.empty:
    print(f"\nFound {len(problematic_prices)} items with price <= 0:")
    print(problematic_prices[['order_id', 'product_id', 'price']].head())
else:
    print("\nNo items found with price <= 0. Data is consistent.")


No items found with price <= 0. Data is consistent.


In [154]:
problematic_freight = items[items['freight_value'] < 0]
if not problematic_freight.empty:
    print(f"\nFound {len(problematic_freight)} items with freight_value < 0:")
    print(problematic_freight[['order_id', 'product_id', 'freight_value']].head())
else:
    print("\nNo items found with freight_value < 0. Data is consistent.")


No items found with freight_value < 0. Data is consistent.


In [155]:
duplicate_order_items = items[items.duplicated(subset=['order_id', 'order_item_id'], keep=False)]
if not duplicate_order_items.empty:
    print(f"\nFound {len(duplicate_order_items)} rows with duplicate (order_id, order_item_id) combinations:")
    print(duplicate_order_items[['order_id', 'order_item_id']].head())
else:
    print("\nNo duplicate (order_id, order_item_id) combinations found. Data is consistent.")


No duplicate (order_id, order_item_id) combinations found. Data is consistent.


In [156]:
problematic_payments_value = payments[payments['payment_value'] < 0]
if not problematic_payments_value.empty:
    print(f"\nFound {len(problematic_payments_value)} payments with payment_value < 0:")
    print(problematic_payments_value[['order_id', 'payment_value']].head())
else:
    print("\nNo payments found with payment_value < 0. Data is consistent.")


No payments found with payment_value < 0. Data is consistent.


In [157]:
payments['payment_type'] = payments['payment_type'].str.lower()
print("\n[payment_type] column standardized to lowercase.")
print(payments['payment_type'].value_counts())


[payment_type] column standardized to lowercase.
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [158]:
products['product_category_name'] = products['product_category_name'].str.lower()
print("\n[product_category_name] column standardized to lowercase.")


[product_category_name] column standardized to lowercase.


In [163]:
# Replace 'null category' with 'unknown'
products.loc[products['product_category_name'] == 'null category', 'product_category_name'] = 'unknown'
print("Replaced 'null category' with 'unknown' in product_category_name.")

Replaced 'null category' with 'unknown' in product_category_name.


In [166]:
dimension_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
problematic_dimensions = products[(products[dimension_cols] <= 0).any(axis=1)]
if not problematic_dimensions.empty:
    print(f"Found {len(problematic_dimensions)} products with weight or dimensions <= 0:")
    print(problematic_dimensions[['product_id'] + dimension_cols].head())
else:
    print("No products found with weight or dimensions <= 0. Data is consistent.")

Found 4 products with weight or dimensions <= 0:
                             product_id  product_weight_g  product_length_cm  \
9769   81781c0fed9fe1ad6e8c81fca1e1cb08               0.0               30.0   
13683  8038040ee2a71048d4bdbbdc985b69ab               0.0               30.0   
14997  36ba42dd187055e1fbe943b2d11430ca               0.0               30.0   
32079  e673e90efa65a5409ff4196c038bb5af               0.0               30.0   

       product_height_cm  product_width_cm  
9769                25.0              30.0  
13683               25.0              30.0  
14997               25.0              30.0  
32079               25.0              30.0  


In [167]:
orders.to_csv('/content/drive/MyDrive/original/orders_clean.csv', index=False)
customers.to_csv('/content/drive/MyDrive/original/customers_clean.csv', index=False)
items.to_csv('/content/drive/MyDrive/original/items_clean.csv', index=False)
payments.to_csv('/content/drive/MyDrive/original/payments_clean.csv', index=False)
reviews.to_csv('/content/drive/MyDrive/original/reviews_clean.csv', index=False)
geolocations.to_csv('/content/drive/MyDrive/original/geolocations_clean.csv', index=False)
sellers.to_csv('/content/drive/MyDrive/original/sellers_clean.csv', index=False)
products.to_csv('/content/drive/MyDrive/original/products_clean.csv', index=False)
name_translations.to_csv('/content/drive/MyDrive/original/name_translations_clean.csv', index=False)

## Key Takeaways from Data Cleaning and Preprocessing

### 1️- Data Type Conversions
- Essential date columns in **orders**, **items**, and **reviews** were converted to `datetime` objects, enabling time-series analysis.
  - Orders: `order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, `order_estimated_delivery_date`
  - Items: `shipping_limit_date`
  - Reviews: `review_creation_date`, `review_answer_timestamp`

### 2️- Missing Value Handling
- Rows with missing values in **orders** and **products** were removed for completeness.
- Highly sparse columns in **reviews** (`review_comment_title`, `review_comment_message`) were dropped to reduce noise.

### 3️- Order Status Filtering
- **Canceled** and **unavailable** orders were removed from **orders**, focusing on successful or ongoing transactions.

### 4️- Data Consistency Checks
- Date consistency: delivery ≥ purchase, approved ≥ purchase → all passed.
- No invalid prices (`price <= 0`) or freight values (`freight_value < 0`) in **items**.
- No duplicate `(order_id, order_item_id)` in **items**.
- No negative `payment_value` in **payments**.

### 5️- Data Standardization
- `payment_type` (**payments**) and `product_category_name` (**products**) converted to lowercase.
- Null categories in `product_category_name` replaced with `"unknown"`.

### 6️- Product Dimensions
- Found 4 products with `product_weight_g` or dimensions ≤ 0 — may need further review.

### 7️- Duplicate ID Analysis
- Expected duplicates: `customer_unique_id`, `order_id` (items/payments), `product_id`, `seller_id`, `order_id` (reviews)
- Potential issue: duplicate `review_id` if meant to be unique.
- `product_width_cm` flagged mistakenly as duplicate — not an ID.

### 8️- Cleaned Data Export
- All cleaned DataFrames successfully exported to `/content/drive/MyDrive/original/`, ready for next analysis steps.